# Letter metadata with an LLM on the DHinfra cluster

Ten transcribed letters (German, French, English, Italian) from the Hammer-Purgstall
correspondence (1794-1856) go in. For each letter the model returns four values as JSON: the
writer (forename, surname, name particle, profession), the date (`YYYY-MM-DD`), the place of
writing (with a country code) and the language. These are the values a letter needs to be
listed in [correspSearch](https://correspsearch.net) through the CMIF format.

The letters are plain text. Datelines and signatures are part of the running text, and some
letters have neither. The prompt therefore asks the model to answer `unk` where the letter does
not give a value, instead of guessing.

For these ten letters the metadata of the edition serve as a reference, so section 5 can
compare the answers with it. No GPU is needed in your own session: the model runs on the cluster.

**Steps:** 1. set up, 2. look at the prompt, 3. one letter, 4. all ten letters,
5. compare with the reference.

## 1. Set up

In [ ]:
# Run once per session. The cluster's Python environment may already have these.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

Your project token is read from `DHINFRA_KEY`, either from the environment or from a `.env`
file next to this notebook (`DHINFRA_KEY="..."`). Never commit the `.env` file.

In [ ]:
import json
from pathlib import Path
import concurrent.futures as cf

import pandas as pd

import helpers

MODEL = "deepseek-v4.1-flash"   # pick a name from the list printed below
THINKING = False                # True: the model reasons first (slower, more tokens)
WORKERS = 4                     # parallel requests; keep this small, the cluster is shared

PROMPT_DIR = Path("prompts/extraction")
LETTER_DIR = Path("data/letters")
REFERENCE_FILE = Path("data/reference.json")
OUTPUT_DIR = Path("output") / MODEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = helpers.get_client()
# Optional: list the models the API offers, to check that MODEL is spelled correctly.
print([m.id for m in client.models.list().data])

## 2. The prompt

Three files make up the *system prompt*: `prompt.txt` (the task), `encoding_rules.txt` (how to
write dates, places and names, and when to answer `unk`) and `few_shot_examples.txt` (four
letters with the expected JSON). The *user message* is the letter text and nothing else.

In [ ]:
system_prompt = helpers.build_system_prompt(PROMPT_DIR)
print(f"{len(system_prompt):,} characters")
print(system_prompt[:2500], "...")

## 3. One letter

`letter1385.txt` was written in Munich in December 1812 and is signed "Dr. Frank". The signature
gives only the surname, so the rules expect `unk` for the forename unless the letter says more.

In [ ]:
letter_text = (LETTER_DIR / "letter1385.txt").read_text(encoding="utf-8")
print(letter_text)

In [ ]:
raw, usage = helpers.ask_model(client, MODEL, system_prompt, letter_text, thinking=THINKING)
print(raw)
print()
print("tokens in/out:", usage.prompt_tokens, usage.completion_tokens)

## 4. All ten letters

One request per letter, a few at a time. The answers are saved to
`output/<model>/extraction.json`, one entry per letter.

In [ ]:
def run_one(path):
    raw, usage = helpers.ask_model(
        client, MODEL, system_prompt, path.read_text(encoding="utf-8"), thinking=THINKING)
    return path.stem, raw, usage.prompt_tokens, usage.completion_tokens

letters = sorted(LETTER_DIR.glob("*.txt"))
answers = {}
with cf.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(run_one, p): p for p in letters}
    for future in cf.as_completed(futures):
        name = futures[future].stem
        try:
            answers[name] = future.result()
            print("done  ", name)
        except Exception as err:   # keep going; one failed letter should not stop the batch
            print("FAILED", name, "-", err)

saved = {name: helpers.parse_json_answer(raw) for name, (_, raw, _, _) in sorted(answers.items())}
(OUTPUT_DIR / "extraction.json").write_text(
    json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8")
print(sum(a is not None for a in saved.values()), "of", len(saved), "answers are valid JSON")

## 5. Compare with the reference

`data/reference.json` holds, for each letter, the writer, date, place and language from the
metadata of the edition. It also says whether the letter itself gives each value
(`in_letter`): the edition knows who wrote a letter without a signature, the letter text does not.
That decides what counts as a correct `unk`.

- **language**: compared exactly.
- **date**: compared exactly (`1812-12-07` against `1812-12-07`).
- **writer**: compared by surname only. This is a rough check: it does not look at forenames.
- **place**: shown, not scored. The model writes today's English name (`Munich`), the reference
  the German name the edition uses (`München`). Notebook 2 compares places by their GeoNames ID.

Each date and writer gets one of five outcomes:

| outcome | meaning |
|---|---|
| correct | the value matches the reference |
| correct (not in letter) | the model said `unk`, and the letter does not give the value |
| missed | the model said `unk`, but the letter gives the value |
| wrong | the model gave a value that does not match |
| check (not in letter) | the model gave a value the letter does not contain: inferred from context, or made up. Read it. |

In [ ]:
reference = {r["letter"]: r for r in json.loads(REFERENCE_FILE.read_text(encoding="utf-8"))}
rows = [helpers.score_extraction(saved.get(name), reference[name]) for name in sorted(reference)]
table = pd.DataFrame(rows).set_index("letter")
pd.set_option("display.max_colwidth", 60)
table

In [ ]:
print("language:", int(table.language_ok.sum()), "of", len(table), "correct")
print()
print("date:\n" + table.date_result.value_counts().to_string())
print()
print("writer:\n" + table.writer_result.value_counts().to_string())

Letters worth a closer look:

- `letter2967`, St. Petersburg: the dateline gives 30 December 1823 in the Julian calendar, which
  was used in Russia. The reference has the Gregorian date, 11 January 1824. The rules ask for the
  Gregorian date only "if Julian and Gregorian dates are present in the letter"; here only one is.
- `letter1520`: the dateline reads "June 7 London", without a year.
- `letter1067` and `letter4107`: no signature, or only an initial. The edition knows the writers,
  the letter text does not name them.

In [ ]:
# One answer next to the letter. Change the name to see another one.
name = "letter2967"
print((LETTER_DIR / f"{name}.txt").read_text(encoding="utf-8")[-400:])
print("-" * 60)
print(json.dumps(saved.get(name), ensure_ascii=False, indent=2))

## Where next

- Change `MODEL`, set `THINKING = True`, or change the rules in `prompts/extraction/`, and run
  again. The output goes to a folder named after the model.
- Put your own plain-text letters into `data/letters/`. Without a reference entry, section 5 has
  nothing to compare with; the answers are still saved.
- `02_link_authorities.ipynb` takes the next step: GND and GeoNames IDs, and CMIF.